<a href="https://colab.research.google.com/github/osacr6/SC707_Proyecto_G6/blob/Daniel/SC707_Proyecto_G6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grupo 6 · SC-707
## Big Data - Universidad Fidélitas
## Proyecto final - Avance 2
#### Perfiles de los cantones de Costa Rica según sus pymes activas

- Cambios en la cantidad de pymes del cantón entre 2020 y 2026 (Crecimiento).
- PCT de las pymes que siguen en el registro desde el 2020 (Permanencia)

**Datos:** cada registro es una pyme en un año y un cantón (lista de pymes activas del MEIC, enero de 2020 a 2026), más la población de cada cantón.

**Subir todos los archivos en `MyDrive/sc-707-pymes`**.

# 1 - Datos

In [15]:
from google.colab import drive
drive.mount('/content/drive')


import os, re, unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Subir archivos a Drive
DATA_DIR = BASE_DIR = '/content/drive/MyDrive/sc-707-pymes'
AÑOS = range(2020, 2027)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Unir archivos

- Cada año trae columnas con otros nombres, en otro orden o faltan.
- Se conservan las que existen en todos los años

In [16]:
REGLAS = [
  ('IDENTIFICACI', 'ID'),
  ('TAMA', 'TAMANO'),
  ('PROVINCIA', 'PROVINCIA'),
  ('CANT', 'CANTON'),
  ('REGI', 'REGION'),
  ('ACTIVIDAD CIIU', 'CIIU'),
  ('ACTIVIDAD PRIMARIA', 'CIIU')
]

def _norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s).strip().upper()

def renombrar(cols):
    nuevo = {}
    for c in cols:
        for prefijo, nombre in REGLAS:
            if _norm(c).startswith(prefijo) and nombre not in nuevo.values():
                nuevo[c] = nombre
                break
    return nuevo

estructura, partes = [], []
for anio in AÑOS:
    crudo = pd.read_csv(os.path.join(DATA_DIR, f'{anio}-01.csv'), encoding='latin-1', dtype=str)
    crudo = crudo.loc[:, ~crudo.columns.str.startswith('Unnamed')].dropna(how='all')
    mapa = renombrar(crudo.columns)
    t = crudo.rename(columns=mapa)[list(mapa.values())].copy()
    t['ANIO'] = anio
    estructura.append({'Año': anio, 'Registros': len(t), 'Columnas': crudo.shape[1]})
    partes.append(t)

serie = pd.concat(partes, ignore_index=True)
for c in ['ID', 'TAMANO', 'PROVINCIA', 'CANTON', 'REGION', 'CIIU']:
    serie[c] = serie[c].astype('string').str.strip()

estructura = pd.DataFrame(estructura)

estructura

,Año,Registros,Columnas
0,2020,19923,12
1,2021,24038,11
2,2022,25691,11
3,2023,21644,12
4,2024,25606,12
5,2025,25661,9
6,2026,31757,13


## 1.2 Limpieza

In [11]:
reporte = {}


# Identificación limpia: mayúsculas, sin espacios ni guiones; si es solo números, sin ceros a la izquierda
ide = serie['ID'].fillna('').str.upper().str.replace(r'[^0-9A-Z]', '', regex=True)
serie['ID'] = ide.where(~ide.str.fullmatch(r'\d+'), ide.str.lstrip('0'))
serie = serie[serie['ID'] != '']

dup = serie.duplicated(subset=['ANIO', 'ID'])
reporte['Identificación repetida'] = serie[dup].groupby('ANIO').size()
serie = serie[~dup]

serie['CIIU_COD'] = serie['CIIU'].str.extract(r'^\s*(\d{4})', expand=False)
reporte['Sin código CIIU'] = serie[serie['CIIU_COD'].isna()].groupby('ANIO').size()

pd.DataFrame(reporte).fillna(0).astype(int).rename_axis('Año')

,Identificación repetida,Sin código CIIU
Año,,
2020,0,12
2021,0,18
2022,0,15
2023,0,9
2024,0,15
2025,0,16
2026,0,30


#2 - Análisis exploratorio

## 2.1 Tamaño de las empresas


In [17]:
(pd.crosstab(serie['ANIO'], serie['TAMANO'], normalize='index') * 100).round(1)

TAMANO,Mediana,Micro,Pequeña
ANIO,,,
2020,4.2,77.3,18.4
2021,3.8,79.6,16.6
2022,2.8,84.0,13.3
2023,1.8,88.2,10.0
2024,1.2,89.9,8.9
2025,0.4,90.1,9.4
2026,0.4,90.9,8.7


## 2.2 Actividad económica



In [14]:
# Cada pyme trae un código CIIU de 4 dígitos, por ejemplo 5610 - Actividades de restaurantes…
SECCIONES = [
    ( 1,  3, 'A', 'Agropecuario'),
    ( 5,  9, 'B', 'Minas'),
    (10, 33, 'C', 'Manufactura'),
    (35, 35, 'D', 'Electricidad'),
    (36, 39, 'E', 'Agua y residuos'),
    (41, 43, 'F', 'Construcción'),
    (45, 47, 'G', 'Comercio'),
    (49, 53, 'H', 'Transporte'),
    (55, 56, 'I', 'Alojamiento y comidas'),
    (58, 63, 'J', 'Información y comunicación'),
    (64, 66, 'K', 'Financieras'),
    (68, 68, 'L', 'Inmobiliarias'),
    (69, 75, 'M', 'Profesionales'),
    (77, 82, 'N', 'Administrativos'),
    (84, 84, 'O', 'Administración pública'),
    (85, 85, 'P', 'Enseñanza'),
    (86, 88, 'Q', 'Salud'),
    (90, 93, 'R', 'Artes y recreación'),
    (94, 96, 'S', 'Otros servicios'),
    (97, 98, 'T', 'Hogares'),
    (99, 99, 'U', 'Extraterritoriales'),
]
NOMBRE_SECCION = {s: n for _, _, s, n in SECCIONES}

def seccion(cod):
    if pd.isna(cod): return 'Sin código'
    d = int(str(cod)[:2])
    return next((s for a, b, s, _ in SECCIONES if a <= d <= b), 'Sin código')

serie['SECCION'] = serie['CIIU_COD'].map(seccion)
comp_anio = (pd.crosstab(serie['SECCION'], serie['ANIO'], normalize='columns') * 100).round(1)
comp_anio.index = [f'{s} · {NOMBRE_SECCION.get(s, s)}' for s in comp_anio.index]
comp_anio.sort_values(2026, ascending=False).head(10)

ANIO,2020,2021,2022,2023,2024,2025,2026
G · Comercio,34.0,33.2,32.2,32.1,31.7,30.8,29.6
M · Profesionales,12.2,12.4,13.0,13.7,13.6,13.6,13.7
C · Manufactura,10.0,10.2,10.8,11.0,11.3,10.9,10.7
I · Alojamiento y comidas,8.6,8.8,8.6,8.1,8.6,9.3,9.4
Q · Salud,7.2,7.1,6.8,7.2,6.9,6.8,7.0
N · Administrativos,5.4,5.6,5.7,5.6,5.4,5.2,5.5
F · Construcción,3.8,4.3,4.7,4.9,5.1,5.3,5.4
H · Transporte,4.2,4.3,4.4,4.0,4.4,4.7,4.7
S · Otros servicios,6.1,5.6,5.2,5.1,4.2,4.3,4.5
J · Información y comunicación,2.8,3.1,3.3,3.1,3.2,3.1,3.2


## 2.3 Tabla de frecuencia